# 📖 Lab 3: HTTP 429 Response & Complete Middleware

When a request exceeds the rate limit, we need to decide: **drop or queue?** And when we drop, what information do we give the client so they can recover gracefully?

## Drop vs Queue

| Strategy | Pros | Cons |
|----------|------|------|
| **Drop (HTTP 429)** ✅ | Predictable latency, no memory buildup, client knows immediately | Client must handle retry logic |
| **Queue** | Requests eventually processed | Memory pressure, unpredictable latency, retry storms, complex |

For interactive APIs: **always drop**. Return 429 with helpful headers so clients can back off intelligently.

## The Response Headers

```
HTTP/1.1 429 Too Many Requests
X-RateLimit-Limit: 100           ← the ceiling
X-RateLimit-Remaining: 0         ← how many left
X-RateLimit-Reset: 1640995200    ← when limit resets (Unix timestamp)
Retry-After: 60                  ← seconds to wait

{"error": "Rate limit exceeded", "message": "Try again in 60 seconds."}
```

## Learning Objectives

- Build a complete rate limiter middleware (identification → algorithm → response)
- Generate proper HTTP 429 responses with rate limit headers
- Simulate a client that respects `Retry-After` headers
- See the full end-to-end flow: request → check → allow/reject with headers

## 🛠️ Setup

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

Select the **"Rate Limiter (Python)"** kernel.

In [ ]:
import redis
import time
import math
import json
from dataclasses import dataclass, field

redis_client = redis.Redis(host="localhost", port=6381, decode_responses=True)
redis_client.flushdb()
print(f"✅ Redis: {'connected' if redis_client.ping() else 'FAILED'}")

## 🔧 Complete Rate Limiter Middleware

Let's put everything together from Labs 1-3 into a single middleware that:
1. Extracts client identity from the request
2. Runs the Token Bucket algorithm via Redis Lua
3. Returns either the response (200) or a proper 429 with headers

In [ ]:
# ── Token Bucket Lua Script (from Lab 2) ──

TOKEN_BUCKET_LUA = """
local key = KEYS[1]
local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])
local ttl = tonumber(ARGV[4])

local tokens = tonumber(redis.call('HGET', key, 'tokens') or capacity)
local last_refill = tonumber(redis.call('HGET', key, 'last_refill') or now)

local elapsed = now - last_refill
tokens = math.min(capacity, tokens + elapsed * refill_rate)

local allowed = 0
local remaining = math.floor(tokens)

if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
    remaining = math.floor(tokens)
end

redis.call('HSET', key, 'tokens', tostring(tokens))
redis.call('HSET', key, 'last_refill', tostring(now))
redis.call('EXPIRE', key, ttl)

return {allowed, remaining}
"""

token_bucket_script = redis_client.register_script(TOKEN_BUCKET_LUA)


# ── HTTP Request / Response models ──

@dataclass
class HttpRequest:
    method: str
    path: str
    headers: dict[str, str]
    remote_ip: str


@dataclass
class HttpResponse:
    status_code: int
    headers: dict[str, str]
    body: dict


# ── Rate Limiter Middleware ──

@dataclass
class RateLimitConfig:
    capacity: int = 10
    refill_rate: float = 2.0
    ttl: int = 3600


def rate_limit_middleware(request: HttpRequest, config: RateLimitConfig = RateLimitConfig()) -> HttpResponse:
    """
    Complete rate limiter middleware. Returns:
    - 200 with rate limit headers if allowed
    - 429 with rate limit headers + Retry-After if blocked
    """
    # Step 1: Extract client ID (prefer user > API key > IP)
    client_id = (
        request.headers.get("X-User-Id")
        or request.headers.get("X-API-Key")
        or request.headers.get("X-Forwarded-For", request.remote_ip).split(",")[0].strip()
    )

    # Step 2: Check rate limit via Redis Lua script
    key = f"rl:{client_id}"
    now = time.time()
    result = token_bucket_script(
        keys=[key],
        args=[config.capacity, config.refill_rate, now, config.ttl],
    )
    allowed = bool(result[0])
    remaining = result[1]

    # Step 3: Calculate reset time
    if remaining == 0 and not allowed:
        tokens_needed = 1
        seconds_until_token = tokens_needed / config.refill_rate
        retry_after = math.ceil(seconds_until_token)
        reset_time = int(now + seconds_until_token)
    else:
        retry_after = 0
        reset_time = int(now + (config.capacity - remaining) / config.refill_rate)

    # Step 4: Build rate limit headers (always included, even on 200)
    rl_headers = {
        "X-RateLimit-Limit": str(config.capacity),
        "X-RateLimit-Remaining": str(remaining),
        "X-RateLimit-Reset": str(reset_time),
    }

    if allowed:
        return HttpResponse(
            status_code=200,
            headers=rl_headers,
            body={"message": "Request processed successfully"},
        )
    else:
        rl_headers["Retry-After"] = str(retry_after)
        return HttpResponse(
            status_code=429,
            headers=rl_headers,
            body={
                "error": "Rate limit exceeded",
                "message": f"You have exceeded the rate limit of {config.capacity} requests. Try again in {retry_after} seconds.",
            },
        )


print("✅ Rate limiter middleware defined.")

## 🧪 Simulating API Traffic

Let's send requests through the middleware and see both 200 and 429 responses with their headers.

In [ ]:
redis_client.flushdb()

config = RateLimitConfig(capacity=5, refill_rate=1.0)  # 5 burst, 1/sec sustained

print(f"🎯 Config: capacity={config.capacity}, refill={config.refill_rate}/sec\n")
print(f"{'#':<4} {'Status':<8} {'Remaining':<12} {'Retry-After':<14} {'Body'}")
print(f"{'─'*70}")

for i in range(8):
    req = HttpRequest("GET", "/api/timeline", {"X-User-Id": "alice"}, "1.2.3.4")
    resp = rate_limit_middleware(req, config)

    retry = resp.headers.get("Retry-After", "—")
    remaining = resp.headers["X-RateLimit-Remaining"]
    body_msg = resp.body.get("message", "")[:40]

    icon = "✅" if resp.status_code == 200 else "❌"
    print(f"{i+1:<4} {icon} {resp.status_code:<5} {remaining:<12} {retry:<14} {body_msg}")

# Show full 429 response
print(f"\n📋 Full 429 response (last request):")
print(f"  Status: {resp.status_code}")
print(f"  Headers:")
for k, v in resp.headers.items():
    print(f"    {k}: {v}")
print(f"  Body: {json.dumps(resp.body, indent=4)}")

## 🤖 Smart Client: Respecting Retry-After

A well-behaved client reads the `Retry-After` header and waits before retrying. This prevents retry storms — where blocked clients immediately retry, creating more load.

In [ ]:
redis_client.flushdb()
config = RateLimitConfig(capacity=3, refill_rate=1.0)

def smart_client(client_id: str, total_requests: int):
    """A well-behaved client that respects Retry-After headers."""
    successful = 0
    retried = 0

    for i in range(total_requests):
        req = HttpRequest("POST", "/api/tweet", {"X-User-Id": client_id}, "1.2.3.4")
        resp = rate_limit_middleware(req, config)

        if resp.status_code == 200:
            successful += 1
            remaining = resp.headers["X-RateLimit-Remaining"]
            print(f"  ✅ Request {i+1}: 200 OK (remaining: {remaining})")
        else:
            retry_after = int(resp.headers["Retry-After"])
            print(f"  ❌ Request {i+1}: 429 — waiting {retry_after}s (Retry-After)...")
            time.sleep(retry_after)
            retried += 1
            # Retry after waiting
            resp = rate_limit_middleware(req, config)
            if resp.status_code == 200:
                successful += 1
                print(f"  ✅ Request {i+1} (retry): 200 OK!")
            else:
                print(f"  ❌ Request {i+1} (retry): still blocked")

    return successful, retried


print(f"🤖 Smart Client — respects Retry-After headers\n")
print(f"  Config: capacity={config.capacity}, refill={config.refill_rate}/sec\n")

successful, retried = smart_client("bob", 6)

print(f"\n  📊 Results: {successful} successful, {retried} retries")
print(f"     A smart client backs off when told → all requests eventually succeed.")
print(f"     A dumb client would hammer the API → all retries fail → wasted load.")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cleaned up.")

## 🏗️ Complete Architecture After Labs 1-3

```
┌────────┐       ┌──────────────────────────────────────────────────┐
│        │──────>│  API Gateway                                     │
│        │       │                                                  │
│ Client │       │  1. Extract client ID (JWT / API key / IP)       │
│        │       │  2. Token Bucket check via Redis Lua (atomic)    │
│        │<──── │  3a. ✅ 200 + X-RateLimit-* headers → App Server │
│        │ 429   │  3b. ❌ 429 + Retry-After header → reject       │
└────────┘       └──────────────────────┬───────────────────────────┘
                                        │
                                        v
                                 ┌──────────────┐
                                 │    Redis     │
                                 │  Lua script  │
                                 │  (atomic)    │
                                 │              │
                                 │ rl:alice     │
                                 │ {tokens: 3,  │
                                 │  last_refill} │
                                 │ EXPIRE 1h    │
                                 └──────────────┘
```

## ✅ Summary

### All 3 Functional Requirements Complete

| # | Requirement | How |
|---|-------------|-----|
| 1 | **Identify clients** | Extract user ID / API key / IP from HTTP headers at API Gateway |
| 2 | **Limit requests by configurable rules** | Token Bucket algorithm in Redis with Lua scripting (atomic) |
| 3 | **Reject with 429 + helpful headers** | `X-RateLimit-Limit`, `Remaining`, `Reset`, `Retry-After` |

### The Complete Middleware Flow

```
Request → Extract client ID → Redis Lua (Token Bucket) → Allow? → 200 + headers
                                                         Reject? → 429 + Retry-After
```

### Key Design Decisions

| Decision | Why |
|----------|-----|
| **Drop, don't queue** | Predictable latency, no memory buildup, client handles retry |
| **Always include rate limit headers** | Even on 200 — clients can monitor their usage proactively |
| **Retry-After header** | Prevents retry storms — smart clients back off instead of hammering |
| **Token Bucket in Redis** | Centralized state across all gateways, atomic Lua script prevents races |

**Next up:** Deep dives — scaling Redis, handling failures (fail open vs closed), multi-region rate limiting